# Smoking is bad

In [ ]:
# TODO: install all the necessary packages

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data
from pathlib import Path

import pandas as pd
import numpy as np

data_path = Path("data") / "smoking_data.csv"
df = pd.read_csv(data_path, index_col=0)
df.head()

In [ ]:
# TODO: what are the dimensions of the data frame?
df.shape

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string
# columns?
df.describe()
# the outcome, smoker and gender columns are not included in the summary
# statistics because they are string columns.
# The .describe() method only provides summary statistics for numeric columns by
# default.

In [ ]:
# TODO: do not forget to check the null values
df.isna().sum()

In [ ]:
# TODO: prepare your data frame so that you can work with
df = df.rename(columns={"smoker": "smokes", "outcome": "alive"})

df["smokes"] = df["smokes"].map({"Yes": 1, "No": 0})
df["alive"] = df["alive"].map({"Alive": 1, "Dead": 0})

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set
# instructions.
df.groupby(["smokes"]).agg(prob=("alive", np.mean)).round(3)

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
# Linear probability model: alive (1/0) on smoking, age, and gender.
# Age enters linearly; C(gender) treats gender as a category.
# HC1 provides heteroskedasticity-robust standard errors.
# The smoking coefficient times 100 is a percentage-point difference.
# This adjusted association does not by itself identify a causal effect.
import statsmodels.formula.api as smf

model_age_gender = smf.ols(
    "alive ~ smokes + age + C(gender)", data=df, missing="raise"
).fit(cov_type="HC1")
print(
    "Smoking coefficient (percentage points):",
    round(100 * model_age_gender.params["smokes"], 3),
)
model_age_gender.summary()

In [ ]:
# Age distributions: all observations, comparing smokers with non-smokers.
# Percentages use each plotted subgroup as its own denominator.
import matplotlib.pyplot as plt
import numpy as np

# Derive shared five-year bins from the full sample, including alive and dead.
age_min = 5 * np.floor(df["age"].min() / 5)
age_max = 5 * np.floor(df["age"].max() / 5) + 5
age_bins = np.arange(age_min, age_max + 5, 5)

# Use the same percentage scale across all six histograms.
comparison_groups = [
    df,
    df[df["gender"] == "male"],
    df[df["gender"] == "female"],
]
max_bin_percent = 0.0
for group in comparison_groups:
    for status in [1, 0]:
        group_ages = group.loc[group["smokes"] == status, "age"].dropna()
        if len(group_ages):
            counts, _ = np.histogram(group_ages, bins=age_bins)
            max_bin_percent = max(
                max_bin_percent, 100 * counts.max() / len(group_ages)
            )
age_y_limit = max(5, 5 * np.ceil(1.1 * max_bin_percent / 5))

plot_data = df
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, sharey=True)
fig.suptitle("Age distributions: All observations", fontsize=14)

for ax, status, title, colour in zip(
    axes, [1, 0], ["Smokers", "Non-smokers"], ["#0072B2", "#D55E00"]
):
    ages = plot_data.loc[plot_data["smokes"] == status, "age"].dropna()
    if len(ages):
        # Weights make the bars sum to 100% within this subgroup.
        ax.hist(
            ages,
            bins=age_bins,
            weights=np.full(len(ages), 100 / len(ages)),
            color=colour,
            edgecolor="white",
        )
    else:
        ax.text(
            0.5,
            0.5,
            "No observations",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )
    ax.set_title(f"{title} (n={len(ages):,})", loc="left")
    ax.set_ylabel("Percentage of group")
    ax.set_xlim(age_min, age_max)
    ax.set_ylim(0, age_y_limit)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

axes[-1].set_xlabel("Age (years)")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

In [ ]:
# Overlay age distributions; percentages include both alive and dead people.
import matplotlib.pyplot as plt
import numpy as np

bins = np.arange(5 * (df["age"].min() // 5), 5 * (df["age"].max() // 5) + 10, 5)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True, sharey=True)
for ax, gender, title in zip(
    axes, [None, "male", "female"], ["Everyone", "Men", "Women"]
):
    group = df if gender is None else df[df["gender"] == gender]
    for status, label, colour in [
        (1, "Smokers", "#0072B2"),
        (0, "Non-smokers", "#D55E00"),
    ]:
        ages = group.loc[group["smokes"] == status, "age"].dropna()
        if len(ages):
            ax.hist(
                ages,
                bins=bins,
                weights=np.full(len(ages), 100 / len(ages)),
                color=colour,
                alpha=0.45,
                label=label,
                linewidth=0,
            )
    ax.set(title=title, xlabel="Age (years)", xlim=(bins[0], bins[-1]))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)
axes[0].set_ylabel("Percentage of group")
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncol=2,
    frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()

In [ ]:
# Proportion alive by age band and smoking status; n labels show group sizes.
# Small groups give noisy proportions; these comparisons are descriptive.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fixed ten-year bands; include alive and dead observations.
band_edges = np.arange(
    10 * (df["age"].min() // 10), 10 * (df["age"].max() // 10) + 20, 10
)
plot_df = df.assign(age_band=pd.cut(df["age"], band_edges, right=False))


def plot_survival(ax: plt.Axes, data: pd.DataFrame, title: str) -> None:
    for status, label, colour in [
        (1, "Smokers", "#0072B2"),
        (0, "Non-smokers", "#D55E00"),
    ]:
        stats = (
            data[data["smokes"] == status]
            .groupby("age_band", observed=False)["alive"]
            .agg(["mean", "count"])
        )
        x = [band.mid for band in stats.index]
        ax.plot(x, 100 * stats["mean"], "o-", color=colour, label=label)
        for age, rate, count in zip(x, stats["mean"], stats["count"]):
            if count:
                ax.annotate(
                    f"n={count}",
                    (age, 100 * rate),
                    xytext=(0, 8 if status else -14),
                    textcoords="offset points",
                    ha="center",
                    fontsize=7,
                    color=colour,
                )
    ax.set(title=title, xlabel="Age band (years)", ylim=(-5, 110))
    ax.set_xticks(
        band_edges[:-1] + 5,
        [f"{int(a)}–{int(a + 9)}" for a in band_edges[:-1]],
        rotation=45,
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.15)


fig, ax = plt.subplots(figsize=(10, 5))
plot_survival(ax, plot_df, "Proportion alive by age and smoking status")
ax.set_ylabel("Alive (%)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

In [ ]:
# Repeat the survival comparison separately for men and women.
# Run the preceding survival cell first to define plot_survival and plot_df.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, gender, title in zip(axes, ["male", "female"], ["Men", "Women"]):
    plot_survival(ax, plot_df[plot_df["gender"] == gender], title)
axes[0].set_ylabel("Alive (%)")
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncol=2,
    frameon=False,
)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

In [ ]:
# Percentage who smoke within each age band, including alive and dead people.
# This denominator is everyone in the age band, not all smokers.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fixed ten-year bands; include alive and dead observations.
band_edges = np.arange(
    10 * (df["age"].min() // 10), 10 * (df["age"].max() // 10) + 20, 10
)
plot_df = df.assign(age_band=pd.cut(df["age"], band_edges, right=False))

smoking_by_age = plot_df.groupby("age_band", observed=False)["smokes"].agg(
    ["mean", "count"]
)
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(smoking_by_age))
bars = ax.bar(x, 100 * smoking_by_age["mean"], color="#0072B2", width=0.75)
ax.bar_label(
    bars,
    labels=[f"n={n}" if n else "No data" for n in smoking_by_age["count"]],
    fontsize=8,
)
ax.set_xticks(
    x, [f"{int(b.left)}–{int(b.right - 1)}" for b in smoking_by_age.index]
)
ax.set(
    title="Smoking prevalence by age",
    xlabel="Age band (years)",
    ylabel="Smoke (%)",
    ylim=(0, 110),
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()